# 7. Comparative Benchmark Environment (CBE)

**Objetivo:** Ejecutar los 4 modelos (Kinetopus, ARIMA, LSTM, Naive) bajo condiciones idénticas de Walk-Forward y guardar todos los resultados en `unified_benchmark_db.csv`.

### Características:
- **Checkpoint/Resume:** Si una combinación (Ticker, Model) ya existe en el CSV, se omite.
- **Reutilización de Datos:** Puede importar resultados existentes de `backtest_kinetopus.csv` y `backtest_arima_.csv`.
- **Esquema Unificado:** Todas las filas incluyen columna `Model` para identificación.

In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Asegurar import desde el root del proyecto
sys.path.insert(0, os.path.dirname(os.getcwd()) if 'notebooks_val' in os.getcwd() else os.getcwd())

from src.ui.market_loader import MarketLoader

## 1. Configuración del Benchmark

In [ ]:
# ============================================================
# CONFIGURACIÓN COMPARTIDA (Igualdad de Condiciones Estricta)
# ============================================================
VENTANA_INICIAL = 150      # Mínimo de velas para entrenar
SALTO = 20                 # Stride entre iteraciones
HORIZONTE = 300            # Velas futuras a predecir
BLOQUES = 60               # Granularidad (cada bloque = 5 velas)
CONTEXT_WINDOW = 1825      # Máximo lookback (~5 años diarios)

# Universo de Activos
TICKERS = ['BTC-USD', 'MSFT', 'XLF']

# Modelos a ejecutar (comentar/descomentar según necesidad)
MODELS_TO_RUN = ['Kinetopus', 'ARIMA', 'LSTM', 'Naive']

# Rutas
DB_PATH = 'notebooks_val/unified_benchmark_db.csv'
KINETOPUS_CSV = 'notebooks_val/backtest_kinetopus.csv'
ARIMA_CSV = 'notebooks_val/backtest_arima_.csv'

# ¿Reutilizar datos existentes?
REUSE_EXISTING_DATA = True

print(f"📋 Configuración del Benchmark:")
print(f"   Ventana Inicial: {VENTANA_INICIAL}")
print(f"   Salto: {SALTO}")
print(f"   Horizonte: {HORIZONTE} velas ({BLOQUES} bloques de {HORIZONTE // BLOQUES})")
print(f"   Context Window: {CONTEXT_WINDOW}")
print(f"   Tickers: {TICKERS}")
print(f"   Modelos: {MODELS_TO_RUN}")
print(f"   Reutilizar existentes: {REUSE_EXISTING_DATA}")

## 2. Sistema de Checkpoint/Resume

In [ ]:
def load_processed_pairs(csv_path: str) -> set:
    """Carga pares (Ticker, Model) ya procesados del CSV unificado."""
    if not os.path.isfile(csv_path):
        return set()
    try:
        df = pd.read_csv(csv_path, usecols=['Ticker', 'Model'])
        return set(zip(df['Ticker'], df['Model']))
    except Exception:
        return set()


def save_results(df_results: pd.DataFrame, csv_path: str, ticker: str, model_name: str):
    """Guarda resultados con checkpoint progresivo (append)."""
    df_save = df_results.copy()
    
    # Asegurar columnas de identificación
    if 'Model' not in df_save.columns:
        df_save.insert(0, 'Model', model_name)
    if 'Ticker' not in df_save.columns:
        df_save.insert(1, 'Ticker', ticker)
    
    file_exists = os.path.isfile(csv_path)
    df_save.to_csv(csv_path, mode='a', header=not file_exists, index=False)
    print(f"   💾 Guardadas {len(df_save)} filas para ({ticker}, {model_name})")


# Cargar pares ya procesados
processed_pairs = load_processed_pairs(DB_PATH)
print(f"\n📂 Pares ya procesados en '{DB_PATH}': {len(processed_pairs)}")
if processed_pairs:
    for pair in sorted(processed_pairs):
        print(f"   ✓ {pair}")

## 3. Reutilización de Datos Existentes (Opcional)

In [ ]:
if REUSE_EXISTING_DATA:
    print("\n🔄 Importando datos existentes...")
    
    # --- Kinetopus ---
    if os.path.isfile(KINETOPUS_CSV) and 'Kinetopus' in MODELS_TO_RUN:
        df_kt = pd.read_csv(KINETOPUS_CSV)
        if 'Ticker' in df_kt.columns:
            for ticker in TICKERS:
                pair = (ticker, 'Kinetopus')
                if pair not in processed_pairs:
                    df_ticker = df_kt[df_kt['Ticker'] == ticker].copy()
                    if len(df_ticker) > 0:
                        df_ticker['Model'] = 'Kinetopus'
                        save_results(df_ticker, DB_PATH, ticker, 'Kinetopus')
                        processed_pairs.add(pair)
                        print(f"   ✅ Importado Kinetopus/{ticker}: {len(df_ticker)} filas")
                    else:
                        print(f"   ⚠️ No hay datos de Kinetopus para {ticker} en {KINETOPUS_CSV}")
        else:
            print(f"   ⚠️ {KINETOPUS_CSV} no tiene columna 'Ticker'")
    
    # --- ARIMA ---
    if os.path.isfile(ARIMA_CSV) and 'ARIMA' in MODELS_TO_RUN:
        df_ar = pd.read_csv(ARIMA_CSV)
        if 'Ticker' in df_ar.columns:
            for ticker in TICKERS:
                pair = (ticker, 'ARIMA')
                if pair not in processed_pairs:
                    df_ticker = df_ar[df_ar['Ticker'] == ticker].copy()
                    if len(df_ticker) > 0:
                        df_ticker['Model'] = 'ARIMA'
                        save_results(df_ticker, DB_PATH, ticker, 'ARIMA')
                        processed_pairs.add(pair)
                        print(f"   ✅ Importado ARIMA/{ticker}: {len(df_ticker)} filas")
                    else:
                        print(f"   ⚠️ No hay datos de ARIMA para {ticker} en {ARIMA_CSV}")
        else:
            print(f"   ⚠️ {ARIMA_CSV} no tiene columna 'Ticker'")
    
    print(f"\n📂 Pares procesados tras importación: {len(processed_pairs)}")
else:
    print("\n⏭️ Reutilización de datos deshabilitada.")

## 4. Instanciación de Evaluadores

In [ ]:
def create_evaluator(model_name: str, df_market: pd.DataFrame, ticker: str):
    """
    Factory que instancia el evaluador correcto por nombre de modelo.
    Retorna (evaluador, éxito: bool).
    """
    try:
        if model_name == 'Kinetopus':
            from src.quant_engine.evaluator import WalkForwardEvaluator
            return WalkForwardEvaluator(df_market, disable_norm=False, disable_returns=False, context_window=CONTEXT_WINDOW), True
        
        elif model_name == 'ARIMA':
            from src.quant_engine.arima_evaluator import AutoARIMAWalkForwardEvaluator
            return AutoARIMAWalkForwardEvaluator(df_market, disable_norm=False, disable_returns=False), True
        
        elif model_name == 'LSTM':
            from src.quant_engine.lstm_evaluator import LSTMWalkForwardEvaluator
            return LSTMWalkForwardEvaluator(df_market, disable_norm=False, disable_returns=False, ticker=ticker, context_window=CONTEXT_WINDOW), True
        
        elif model_name == 'Naive':
            from src.quant_engine.naive_evaluator import NaiveWalkForwardEvaluator
            return NaiveWalkForwardEvaluator(df_market, disable_norm=False, disable_returns=False, context_window=CONTEXT_WINDOW), True
        
        else:
            print(f"   ❌ Modelo desconocido: {model_name}")
            return None, False
    
    except ImportError as e:
        print(f"   ❌ No se pudo importar evaluador para {model_name}: {e}")
        return None, False

print("✅ Factory de evaluadores definida.")

## 5. Ejecución del Benchmark

In [ ]:
print("\n" + "=" * 70)
print("  🚀 EJECUCIÓN DEL COMPARATIVE BENCHMARK ENVIRONMENT")
print("=" * 70)

total_new = 0
skipped = 0
failed = 0

for ticker in TICKERS:
    print(f"\n{'─' * 50}")
    print(f"📈 Activo: {ticker}")
    print(f"{'─' * 50}")
    
    # Descargar datos del mercado (una sola vez por ticker)
    try:
        df_market = MarketLoader.load_ticker_data(ticker, period='10y', interval='1d')
        print(f"   📥 Descargadas {len(df_market)} velas para {ticker}")
    except Exception as e:
        print(f"   ❌ Error descargando datos para {ticker}: {e}")
        continue
    
    if len(df_market) < VENTANA_INICIAL + HORIZONTE:
        print(f"   ⚠️ Historial demasiado corto ({len(df_market)} velas). Mínimo: {VENTANA_INICIAL + HORIZONTE}.")
        continue
    
    for model_name in MODELS_TO_RUN:
        pair = (ticker, model_name)
        
        # Checkpoint: omitir si ya existe
        if pair in processed_pairs:
            print(f"   ⏭️ {model_name}: Ya procesado, omitiendo.")
            skipped += 1
            continue
        
        print(f"\n   🔄 {model_name}: Ejecutando Walk-Forward...")
        
        evaluator, ok = create_evaluator(model_name, df_market, ticker)
        if not ok:
            failed += 1
            continue
        
        t0 = time.time()
        try:
            df_results = evaluator.run(
                initial_window=VENTANA_INICIAL,
                stride=SALTO,
                horizon=HORIZONTE,
                blocks=BLOQUES
            )
            elapsed = time.time() - t0
            
            # Inyectar metadatos si no existen
            if 'Model' not in df_results.columns:
                df_results.insert(0, 'Model', model_name)
            if 'Ticker' not in df_results.columns:
                df_results.insert(1, 'Ticker', ticker)
            
            # Guardar con checkpoint
            save_results(df_results, DB_PATH, ticker, model_name)
            processed_pairs.add(pair)
            total_new += len(df_results)
            
            # Estadísticas rápidas
            valid_pct = (df_results['Validez'] == 'OK').mean() * 100 if 'Validez' in df_results.columns else 100
            print(f"   ✅ {model_name}: {len(df_results)} iteraciones en {elapsed:.1f}s (Validez OK: {valid_pct:.0f}%)")
            
        except Exception as e:
            elapsed = time.time() - t0
            print(f"   ❌ {model_name}: Error tras {elapsed:.1f}s → {e}")
            failed += 1
            continue
        
        # Pausa para limpiar I/O
        time.sleep(0.5)

print(f"\n{'=' * 70}")
print(f"  📊 RESUMEN: {total_new} filas nuevas | {skipped} omitidas | {failed} fallidas")
print(f"{'=' * 70}")

## 6. Sanidad del Dataset Unificado

In [ ]:
if os.path.isfile(DB_PATH):
    df_db = pd.read_csv(DB_PATH)
    print(f"📊 Total Filas en unified_benchmark_db.csv: {len(df_db)}")
    print(f"📊 Total Columnas: {df_db.shape[1]}")
    print(f"\n🏷️ Distribución por Modelo:")
    print(df_db.groupby('Model').size().to_string())
    print(f"\n📈 Distribución por Ticker:")
    print(df_db.groupby('Ticker').size().to_string())
    print(f"\n🔀 Distribución cruzada (Model × Ticker):")
    print(pd.crosstab(df_db['Model'], df_db['Ticker']))
    
    # Validez por modelo
    if 'Validez' in df_db.columns:
        print(f"\n✅ Tasa de Validez por Modelo:")
        validez = df_db.groupby('Model')['Validez'].apply(lambda x: (x == 'OK').mean() * 100)
        for model, pct in validez.items():
            print(f"   {model}: {pct:.1f}% OK")
else:
    print("⚠️ No se encontró unified_benchmark_db.csv")

In [ ]:
if os.path.isfile(DB_PATH):
    df_db = pd.read_csv(DB_PATH)
    display(df_db.head(10))